# PTB-XL Preprocessing v2 — Canonical Pipeline

Built from scratch following peer-reviewed canonical practice:

- **Wagner et al. (2020).** PTB-XL, a large publicly available electrocardiography dataset. *Scientific Data* 7:154. DOI: 10.1038/s41597-020-0495-6
- **Strodthoff et al. (2021).** Deep Learning for ECG Analysis: Benchmarks and Insights from PTB-XL. *IEEE JBHI* 25(5):1519–1528. DOI: 10.1109/JBHI.2020.3022989
- **helme/ecg_ptbxl_benchmarking** reference implementation (Strodthoff et al.'s official code)

## Decisions and citations

| Decision | Choice | Justification |
|---|---|---|
| Sampling rate | 100 Hz (`filename_lr`) | Strodthoff et al. 2021 §III.A |
| AF labels | `AFIB ∪ AFLT`, key presence | Wagner et al. 2020 Table 8; rhythm likelihoods are uniformly 0 in `scp_codes` |
| Patient splits | `strat_fold` from `ptbxl_database.csv` | Wagner et al. 2020; folds 1–8 train / 9 val / 10 test, patient-disjoint |
| Other rhythms (SVARR, SVTAC, PSVT) | Retained in negative class | Mirrors PTB-XL benchmark; matches existing project decision |
| `scp_codes` parsing | `ast.literal_eval` | Safe alternative to `eval()` |
| Quality filter | None at extraction; `validated_by_human` preserved | Decision deferred to downstream |
| Failure logging | Per-record reason + class breakdown | Detects class-conditional bias |
| Sanity checks | Prevalence ≈ 7.27%, record count ≈ 21,799 | Wagner et al. 2020 documented values |

## Output

`data/ptbxl_features.csv` — drop-in replacement for the previous `ptbxl_all_tiers_features.csv`

Schema:
- `ecg_id`, `patient_id`, `strat_fold`, `validated_by_human` (identifiers + canonical metadata)
- 4 Tier 1 demographic features (age, sex, height, weight)
- 5 Tier 2 interval features (heart rate, RR mean, RR std, PR, QRS)
- 5 Tier 2b HRV features (RMSSD, SDNN, pNN50, mean NN, sample entropy) — gold-standard AF-discriminative metrics
- 48 Tier 3 lead-statistic features (12 leads × 4 stats: mean, std, max, min)
- `af_detected` label

Total: **62 features** + identifiers + label.

## 1. Imports and configuration

In [ ]:
import ast
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import wfdb
import neurokit2 as nk
from tqdm import tqdm

import warnings
warnings.filterwarnings('ignore')

# Configuration
PTBXL_DIR = Path('data/ptbxl/')
OUTPUT_PATH = Path('data/ptbxl_features.csv')
FAILURE_LOG_PATH = Path('data/ptbxl_extraction_failures.csv')

SAMPLING_RATE = 100  # Strodthoff et al. 2021 use 100 Hz; filename_lr column
FILENAME_COL = 'filename_lr'  # 'filename_lr' for 100 Hz, 'filename_hr' for 500 Hz

AF_RHYTHM_CODES = {'AFIB', 'AFLT'}  # Wagner et al. 2020 Table 8

EXPECTED_RECORD_COUNT = 21799  # Wagner et al. 2020
EXPECTED_AF_PREVALENCE = 0.0727  # 1587 / 21837 — Wagner et al. 2020 Table 8
PREVALENCE_TOLERANCE = 0.005  # ±0.5 percentage points

LEADS = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF',
         'V1', 'V2', 'V3', 'V4', 'V5', 'V6']

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%H:%M:%S'
)
log = logging.getLogger('ptbxl_prep')

## 2. Label extraction

Per Wagner et al. 2020 supplementary materials, `scp_codes` is a stringified Python dict mapping SCP code → likelihood. **Rhythm statements (including AFIB and AFLT) carry likelihood 0 uniformly** — likelihood is only meaningful for diagnostic statements. Therefore the correct extraction rule is **key presence**, not `value > threshold`.

AF positive class: AFIB (atrial fibrillation, 1,514 records) ∪ AFLT (atrial flutter, 73 records) = 1,587 / 21,799 = **7.27%** prevalence.

We use `ast.literal_eval` (not `eval`) for safety.

In [ ]:
def extract_af_label(scp_codes_str):
    """Extract binary AF label from scp_codes string.
    
    Returns 1 if AFIB or AFLT key is present in scp_codes, else 0.
    Uses key presence (not value threshold) per Wagner et al. 2020.
    """
    if not isinstance(scp_codes_str, str):
        return 0
    try:
        codes = ast.literal_eval(scp_codes_str)
    except (ValueError, SyntaxError):
        return 0
    if not isinstance(codes, dict):
        return 0
    return int(bool(AF_RHYTHM_CODES & set(codes.keys())))


# Quick self-test
_test_cases = [
    ("{'AFIB': 0.0, 'NORM': 0.0}", 1),
    ("{'AFLT': 0.0}", 1),
    ("{'AFIB': 0.0, 'AFLT': 0.0}", 1),
    ("{'NORM': 0.0, 'SR': 0.0}", 0),
    ("{'SVARR': 0.0}", 0),  # supraventricular arrhythmia is NOT AF
    ("{}", 0),
    ("", 0),
    (None, 0),
    ("not a dict", 0),
]
for inp, expected in _test_cases:
    actual = extract_af_label(inp)
    assert actual == expected, f"FAIL: extract_af_label({inp!r}) = {actual}, expected {expected}"
log.info("Label extraction self-test passed (9 cases)")

07:59:05 [INFO] Label extraction self-test passed (9 cases)


## 3. Feature extraction functions

Three tiers:

**Tier 1 — Demographics** (4 features): age, sex, height, weight from `ptbxl_database.csv`.

**Tier 2 — Cardiac intervals** (5 features) from Lead II via NeuroKit2:
- heart rate (mean BPM)
- RR interval mean and std (ms)
- PR interval (ms) — from P-onset to R-onset
- QRS duration (ms) — from R-onset to R-offset

**Tier 2b — Heart Rate Variability** (5 features) from Lead II RR series via NeuroKit2:
- RMSSD (root mean square of successive RR differences) — strongest single AF marker (Tateno & Glass 2001 *Med Biol Eng Comput*)
- SDNN (standard deviation of NN intervals)
- pNN50 (% of successive RRs differing >50 ms)
- mean NN interval
- sample entropy of RR series — irregularity measure

**Tier 3 — Per-lead amplitude statistics** (48 features): mean, std, max, min for each of 12 leads.

Total: 62 features.

In [ ]:
def extract_intervals_and_hrv(signal_lead_ii, fs):
    """Extract cardiac intervals + HRV features from Lead II.
    
    Returns dict with 10 features (5 intervals + 5 HRV).
    All NaN if extraction fails.
    """
    nan_result = {
        'heart_rate': np.nan,
        'rr_interval_mean': np.nan,
        'rr_interval_std': np.nan,
        'pr_interval': np.nan,
        'qrs_duration': np.nan,
        'hrv_rmssd': np.nan,
        'hrv_sdnn': np.nan,
        'hrv_pnn50': np.nan,
        'hrv_mean_nn': np.nan,
        'hrv_sample_entropy': np.nan,
    }
    
    try:
        # Process ECG: cleans signal, finds R-peaks, computes rate
        signals_df, info = nk.ecg_process(signal_lead_ii, sampling_rate=fs)
        rpeaks = info.get('ECG_R_Peaks', np.array([]))
    except Exception as e:
        return nan_result, f'ecg_process: {type(e).__name__}'
    
    out = dict(nan_result)
    
    # Heart rate (mean BPM) — comes from the signals DataFrame as ECG_Rate column
    if 'ECG_Rate' in signals_df.columns:
        rate_series = signals_df['ECG_Rate'].dropna()
        if len(rate_series) > 0:
            out['heart_rate'] = float(rate_series.mean())
    
    # RR intervals (ms)
    if len(rpeaks) >= 2:
        rr_ms = np.diff(rpeaks) / fs * 1000.0
        out['rr_interval_mean'] = float(rr_ms.mean())
        out['rr_interval_std'] = float(rr_ms.std())
        
        # HRV features need at least ~3 RR intervals; require 4 for stability
        if len(rr_ms) >= 4:
            try:
                # Build a synthetic peaks dict for nk.hrv (it expects R-peak indices)
                hrv_time = nk.hrv_time({'ECG_R_Peaks': rpeaks}, sampling_rate=fs)
                if 'HRV_RMSSD' in hrv_time.columns:
                    out['hrv_rmssd'] = float(hrv_time['HRV_RMSSD'].iloc[0])
                if 'HRV_SDNN' in hrv_time.columns:
                    out['hrv_sdnn'] = float(hrv_time['HRV_SDNN'].iloc[0])
                if 'HRV_pNN50' in hrv_time.columns:
                    out['hrv_pnn50'] = float(hrv_time['HRV_pNN50'].iloc[0])
                if 'HRV_MeanNN' in hrv_time.columns:
                    out['hrv_mean_nn'] = float(hrv_time['HRV_MeanNN'].iloc[0])
            except Exception:
                pass  # leave HRV time features as NaN
            
            try:
                # Sample entropy on RR series (signal irregularity)
                samp_en, _ = nk.entropy_sample(rr_ms, dimension=2, tolerance=0.2 * rr_ms.std())
                if not np.isnan(samp_en):
                    out['hrv_sample_entropy'] = float(samp_en)
            except Exception:
                pass
    
    # PR interval and QRS duration via wave delineation
    if len(rpeaks) >= 2:
        try:
            _, waves = nk.ecg_delineate(
                signal_lead_ii, rpeaks, sampling_rate=fs, method='dwt'
            )
            
            p_on = np.array(waves.get('ECG_P_Onsets', []), dtype=float)
            r_on = np.array(waves.get('ECG_R_Onsets', []), dtype=float)
            r_off = np.array(waves.get('ECG_R_Offsets', []), dtype=float)
            
            p_on = p_on[~np.isnan(p_on)]
            r_on = r_on[~np.isnan(r_on)]
            r_off = r_off[~np.isnan(r_off)]
            
            # PR: P-onset to R-onset, paired by index up to min length
            if len(p_on) > 0 and len(r_on) > 0:
                n = min(len(p_on), len(r_on))
                pr_ms = (r_on[:n] - p_on[:n]) / fs * 1000.0
                pr_ms = pr_ms[(pr_ms > 80) & (pr_ms < 320)]  # physiological range
                if len(pr_ms) > 0:
                    out['pr_interval'] = float(pr_ms.mean())
            
            # QRS: R-onset to R-offset
            if len(r_on) > 0 and len(r_off) > 0:
                n = min(len(r_on), len(r_off))
                qrs_ms = (r_off[:n] - r_on[:n]) / fs * 1000.0
                qrs_ms = qrs_ms[(qrs_ms > 40) & (qrs_ms < 200)]  # physiological range
                if len(qrs_ms) > 0:
                    out['qrs_duration'] = float(qrs_ms.mean())
        except Exception:
            pass  # leave PR/QRS as NaN
    
    return out, None

## 4. Main extraction loop

Reads each record's signal file, extracts features, logs failures with per-record reasons.

Failures are logged separately so we can audit class-conditional failure rates afterward (does extraction fail more often on AF records than non-AF? if so, that's selection bias).

In [ ]:
def extract_all(ptbxl_dir, sample_limit=None):
    """Extract features for all PTB-XL records.
    
    Args:
        ptbxl_dir: Path to PTB-XL root (containing ptbxl_database.csv and records100/, records500/)
        sample_limit: If set, only process the first N records (for testing)
    
    Returns:
        (features_df, failures_df)
    """
    ptbxl_dir = Path(ptbxl_dir)
    
    # Load master metadata
    metadata_path = ptbxl_dir / 'ptbxl_database.csv'
    if not metadata_path.exists():
        # Some distributions name it differently
        alt_path = ptbxl_dir / 'ptbxl.csv'
        if alt_path.exists():
            metadata_path = alt_path
    
    log.info(f'Loading metadata from {metadata_path}')
    metadata = pd.read_csv(metadata_path, index_col='ecg_id')
    log.info(f'Loaded {len(metadata)} records')
    
    # Verify required columns
    required_cols = ['patient_id', 'scp_codes', 'strat_fold', FILENAME_COL]
    missing_cols = [c for c in required_cols if c not in metadata.columns]
    if missing_cols:
        raise ValueError(f'Missing required columns: {missing_cols}')
    
    if sample_limit:
        metadata = metadata.head(sample_limit)
        log.info(f'Sample limit: processing first {sample_limit} records')
    
    features_list = []
    failures = []
    
    for ecg_id, row in tqdm(metadata.iterrows(), total=len(metadata), desc='Extracting'):
        # Build feature dict starting with identifiers and metadata
        feature_dict = {
            'ecg_id': ecg_id,
            'patient_id': int(row['patient_id']),
            'strat_fold': int(row['strat_fold']),
            'validated_by_human': bool(row.get('validated_by_human', False)),
        }
        
        # Tier 1: demographics
        feature_dict['age'] = float(row['age']) if pd.notna(row.get('age')) else np.nan
        feature_dict['sex'] = int(row['sex']) if pd.notna(row.get('sex')) else np.nan
        feature_dict['height'] = float(row['height']) if pd.notna(row.get('height')) else np.nan
        feature_dict['weight'] = float(row['weight']) if pd.notna(row.get('weight')) else np.nan
        
        # AF label (always extract, regardless of waveform success)
        af_label = extract_af_label(row.get('scp_codes', ''))
        feature_dict['af_detected'] = af_label
        
        # Tier 2 + Tier 3: waveform features
        record_path = ptbxl_dir / row[FILENAME_COL]
        
        try:
            record = wfdb.rdrecord(str(record_path))
            signals = record.p_signal  # (n_samples, 12)
            fs = record.fs
            
            if signals.shape[1] != 12:
                raise ValueError(f'Expected 12 leads, got {signals.shape[1]}')
            
            # Tier 2 + 2b: Lead II intervals + HRV
            ii_features, ii_err = extract_intervals_and_hrv(signals[:, 1], fs)
            feature_dict.update(ii_features)
            if ii_err:
                failures.append({
                    'ecg_id': ecg_id,
                    'af_detected': af_label,
                    'stage': 'intervals_hrv',
                    'reason': ii_err,
                })
            
            # Tier 3: per-lead amplitude statistics
            feature_dict.update(extract_lead_statistics(signals))
        
        except FileNotFoundError as e:
            failures.append({
                'ecg_id': ecg_id,
                'af_detected': af_label,
                'stage': 'file_load',
                'reason': f'FileNotFound: {record_path}',
            })
            # Fill all waveform features with NaN
            for f in ['heart_rate', 'rr_interval_mean', 'rr_interval_std',
                      'pr_interval', 'qrs_duration',
                      'hrv_rmssd', 'hrv_sdnn', 'hrv_pnn50',
                      'hrv_mean_nn', 'hrv_sample_entropy']:
                feature_dict[f] = np.nan
            for lead in LEADS:
                for stat in ['mean', 'std', 'max', 'min']:
                    feature_dict[f'{lead}_{stat}'] = np.nan
        
        except Exception as e:
            failures.append({
                'ecg_id': ecg_id,
                'af_detected': af_label,
                'stage': 'waveform_processing',
                'reason': f'{type(e).__name__}: {str(e)[:100]}',
            })
            for f in ['heart_rate', 'rr_interval_mean', 'rr_interval_std',
                      'pr_interval', 'qrs_duration',
                      'hrv_rmssd', 'hrv_sdnn', 'hrv_pnn50',
                      'hrv_mean_nn', 'hrv_sample_entropy']:
                feature_dict[f] = np.nan
            for lead in LEADS:
                for stat in ['mean', 'std', 'max', 'min']:
                    feature_dict[f'{lead}_{stat}'] = np.nan
        
        features_list.append(feature_dict)
    
    df = pd.DataFrame(features_list)
    failures_df = pd.DataFrame(failures)
    
    return df, failures_df

## 5. Test on a small sample first

Before committing to a 30+ minute full extraction, validate on 200 records that:
- File paths resolve correctly
- WFDB loading works
- NeuroKit2 returns sensible features
- Schema is as expected

In [ ]:
log.info('=== SMOKE TEST: 200 records ===')
df_test, failures_test = extract_all(PTBXL_DIR, sample_limit=200)

log.info(f'Test DataFrame shape: {df_test.shape}')
log.info(f'Test failures: {len(failures_test)} ({100*len(failures_test)/len(df_test):.1f}%)')

# Schema check
expected_columns = (
    ['ecg_id', 'patient_id', 'strat_fold', 'validated_by_human']
    + ['age', 'sex', 'height', 'weight']
    + ['heart_rate', 'rr_interval_mean', 'rr_interval_std', 'pr_interval', 'qrs_duration']
    + ['hrv_rmssd', 'hrv_sdnn', 'hrv_pnn50', 'hrv_mean_nn', 'hrv_sample_entropy']
    + [f'{l}_{s}' for l in LEADS for s in ['mean', 'std', 'max', 'min']]
    + ['af_detected']
)
missing = set(expected_columns) - set(df_test.columns)
extra = set(df_test.columns) - set(expected_columns)
assert not missing, f'Missing columns: {missing}'
assert not extra, f'Unexpected extra columns: {extra}'
log.info(f'Schema OK: {len(expected_columns)} columns ({len(expected_columns)-6} features + 5 IDs/metadata + 1 label)')

# Spot check: HRV features should be non-trivial for at least some records
rmssd_valid = df_test['hrv_rmssd'].notna().sum()
log.info(f'Records with valid RMSSD: {rmssd_valid}/{len(df_test)}')
if rmssd_valid > 0:
    log.info(f'RMSSD range: {df_test["hrv_rmssd"].min():.1f} – {df_test["hrv_rmssd"].max():.1f} ms')

if len(failures_test) > 0:
    log.info('Failure breakdown by stage:')
    print(failures_test['stage'].value_counts().to_string())
    log.info('Sample failure reasons:')
    print(failures_test['reason'].value_counts().head().to_string())

07:59:05 [INFO] === SMOKE TEST: 200 records ===
07:59:05 [INFO] Loading metadata from data/ptbxl/ptbxl.csv
07:59:05 [INFO] Loaded 21799 records
07:59:05 [INFO] Sample limit: processing first 200 records
Extracting: 100%|████████████████████████████████████████████████████████████| 200/200 [00:09<00:00, 21.40it/s]
07:59:15 [INFO] Test DataFrame shape: (200, 67)
07:59:15 [INFO] Test failures: 1 (0.5%)
07:59:15 [INFO] Schema OK: 67 columns (61 features + 5 IDs/metadata + 1 label)
07:59:15 [INFO] Records with valid RMSSD: 199/200
07:59:15 [INFO] RMSSD range: 5.0 – 475.9 ms
07:59:15 [INFO] Failure breakdown by stage:
07:59:15 [INFO] Sample failure reasons:


stage
intervals_hrv    1
reason
ecg_process: ValueError    1


## 6. Full extraction

If the smoke test looks reasonable (low failure rate, schema correct, HRV features populated), run on the full 21,799 records. This typically takes 30–60 minutes depending on hardware.

In [ ]:
# Gate: only run full extraction if smoke test was healthy
smoke_ok = (
    len(df_test) >= 180
    and len(failures_test) / len(df_test) < 0.10  # <10% catastrophic failures
    and df_test['heart_rate'].notna().sum() > 100  # most should have heart rate
)

assert smoke_ok, (
    'Smoke test failed: high failure rate or too few HR extractions. '
    'Inspect failures_test before proceeding.'
)
log.info('Smoke test passed. Running full extraction...')

df_full, failures_full = extract_all(PTBXL_DIR)

log.info(f'Full extraction complete')
log.info(f'Records processed: {len(df_full)}')
log.info(f'Total failures: {len(failures_full)} ({100*len(failures_full)/len(df_full):.2f}%)')

07:59:15 [INFO] Smoke test passed. Running full extraction...
07:59:15 [INFO] Loading metadata from data/ptbxl/ptbxl.csv
07:59:15 [INFO] Loaded 21799 records
Extracting: 100%|████████████████████████████████████████████████████████| 21799/21799 [19:04<00:00, 19.05it/s]
08:18:19 [INFO] Full extraction complete
08:18:19 [INFO] Records processed: 21799
08:18:19 [INFO] Total failures: 72 (0.33%)


## 7. Sanity checks

Hard assertions on canonical PTB-XL properties. **Failure here means something is fundamentally wrong** — different version of PTB-XL, corrupted download, or schema change in the upstream dataset.

References:
- Wagner et al. 2020, *Scientific Data* 7:154 — documents 21,799 records (post-cleaning) and 1,587 AF cases (7.27% prevalence)
- Project chat history confirms 7.27% on the AFIB ∪ AFLT key-presence rule

In [ ]:
# Record count
n = len(df_full)
log.info(f'Record count: {n} (expected ~{EXPECTED_RECORD_COUNT})')
assert abs(n - EXPECTED_RECORD_COUNT) < 100, (
    f'Record count {n} differs from expected {EXPECTED_RECORD_COUNT} by more than 100. '
    'Check PTB-XL version and ptbxl_database.csv integrity.'
)

# AF prevalence — the load-bearing canonical check
prevalence = df_full['af_detected'].mean()
log.info(f'AF prevalence: {prevalence:.4f} (expected {EXPECTED_AF_PREVALENCE:.4f})')
assert abs(prevalence - EXPECTED_AF_PREVALENCE) < PREVALENCE_TOLERANCE, (
    f'AF prevalence {prevalence:.4f} outside tolerance ±{PREVALENCE_TOLERANCE} '
    f'of canonical {EXPECTED_AF_PREVALENCE:.4f}. Check label extraction logic.'
)

# Patient count — should be ~18,885 unique patients per Wagner 2020
n_patients = df_full['patient_id'].nunique()
log.info(f'Unique patients: {n_patients} (expected ~18,885)')
assert 18000 < n_patients < 19500, f'Patient count {n_patients} outside expected range'

# Fold distribution — folds 1–10, roughly balanced
fold_counts = df_full['strat_fold'].value_counts().sort_index()
log.info(f'Fold distribution:\n{fold_counts.to_string()}')
assert set(fold_counts.index) == set(range(1, 11)), f'Unexpected fold values: {fold_counts.index.tolist()}'

# Stratification check: AF prevalence should be similar across folds (±2%)
fold_prevalences = df_full.groupby('strat_fold')['af_detected'].mean()
log.info(f'AF prevalence by fold:\n{fold_prevalences.round(4).to_string()}')
max_dev = (fold_prevalences - prevalence).abs().max()
assert max_dev < 0.02, f'Folds not properly stratified: max deviation {max_dev:.4f}'

# Class-conditional failure rate check — does extraction fail differentially by class?
if len(failures_full) > 0:
    fail_by_class = failures_full.groupby('af_detected').size()
    af_records = (df_full['af_detected'] == 1).sum()
    nonaf_records = (df_full['af_detected'] == 0).sum()
    af_fail_rate = fail_by_class.get(1, 0) / max(af_records, 1)
    nonaf_fail_rate = fail_by_class.get(0, 0) / max(nonaf_records, 1)
    log.info(f'Failure rate by class: AF={af_fail_rate:.4f}, non-AF={nonaf_fail_rate:.4f}')
    if af_fail_rate > 2 * nonaf_fail_rate:
        log.warning(
            f'AF failure rate ({af_fail_rate:.4f}) is more than 2x non-AF rate ({nonaf_fail_rate:.4f}). '
            'This may introduce class-conditional bias. Investigate before using.'
        )

log.info('All sanity checks passed.')

08:18:19 [INFO] Record count: 21799 (expected ~21799)
08:18:19 [INFO] AF prevalence: 0.0720 (expected 0.0727)
08:18:19 [INFO] Unique patients: 18869 (expected ~18,885)
08:18:19 [INFO] Fold distribution:
strat_fold
1     2175
2     2181
3     2192
4     2174
5     2174
6     2173
7     2176
8     2173
9     2183
10    2198
08:18:19 [INFO] AF prevalence by fold:
strat_fold
1     0.0722
2     0.0715
3     0.0716
4     0.0722
5     0.0727
6     0.0713
7     0.0726
8     0.0732
9     0.0715
10    0.0714
08:18:19 [INFO] Failure rate by class: AF=0.0089, non-AF=0.0029
08:18:19 [WARNING] AF failure rate (0.0089) is more than 2x non-AF rate (0.0029). This may introduce class-conditional bias. Investigate before using.
08:18:19 [INFO] All sanity checks passed.


## 8. Feature summary

In [ ]:
feature_cols = [c for c in df_full.columns
                if c not in ('ecg_id', 'patient_id', 'strat_fold', 'validated_by_human', 'af_detected')]
log.info(f'Total features: {len(feature_cols)}')

# Missing-value report
missing_pct = (df_full[feature_cols].isna().sum() / len(df_full) * 100).sort_values(ascending=False)
log.info('Top 10 features by missing rate:')
print(missing_pct.head(10).round(2).to_string())

high_missing = missing_pct[missing_pct > 20]
if len(high_missing) > 0:
    log.warning(f'{len(high_missing)} features have >20% missing values')

# Validated-by-human breakdown
vbh = df_full['validated_by_human'].value_counts()
log.info(f'validated_by_human distribution:\n{vbh.to_string()}')

08:18:19 [INFO] Total features: 62
08:18:19 [INFO] Top 10 features by missing rate:
08:18:19 [WARNING] 2 features have >20% missing values
08:18:19 [INFO] validated_by_human distribution:
validated_by_human
True     16056
False     5743


height                68.01
weight                56.78
pr_interval           10.95
qrs_duration          10.47
hrv_rmssd              0.37
hrv_sample_entropy     0.37
hrv_pnn50              0.37
hrv_sdnn               0.37
hrv_mean_nn            0.37
rr_interval_std        0.33


## 9. Save outputs

Two files:
- `data/ptbxl_features.csv` — main feature dataset
- `data/ptbxl_extraction_failures.csv` — per-record failure log for audit

**Note on missing values:** unlike the previous preprocessor, this version **does not drop rows**. Downstream code (`paper_utils.py`) decides imputation/dropping. This preserves the full denominator for prevalence calculations.

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_full.to_csv(OUTPUT_PATH, index=False)
log.info(f'Saved features to {OUTPUT_PATH} ({df_full.shape[0]} rows × {df_full.shape[1]} cols)')

if len(failures_full) > 0:
    failures_full.to_csv(FAILURE_LOG_PATH, index=False)
    log.info(f'Saved failure log to {FAILURE_LOG_PATH} ({len(failures_full)} failures)')

log.info('Done.')

08:18:20 [INFO] Saved features to data/ptbxl_features.csv (21799 rows × 67 cols)
08:18:20 [INFO] Saved failure log to data/ptbxl_extraction_failures.csv (72 failures)
08:18:20 [INFO] Done.
